# Define channels & threshold to detect semi-automatically vigilance states

Load packages

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import signal
from scipy import interpolate
from scipy import fftpack
from pathlib import Path
import warnings
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, TimeFreqViewer
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, EpochEncoder
from ephyviewer import InMemoryAnalogSignalSource
from matplotlib import cm
from matplotlib.colors import to_hex
from ephyviewer import AnalogSignalSourceWithScatter
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import ast
warnings.filterwarnings("ignore")

#Load LFP coordinates 
Channels = f'{os.getcwd()}/_LFP_coordinates_of_all_mice.csv'
all_LFPcoordinates = pd.read_csv(Channels, index_col=0, sep=';')



ch_EMG = None
ch_Theta = None

ch_nb_EMG = None
ch_nb_Theta = None

th_EMG = None
th_Theta = None

In [ ]:
try: # tries to retrieve dpath either from a previous run or from a previous notebook
    %store -r dpath
    #dpath=dpath.parent
except:
    print("the path was not defined in store")
    dpath = "//10.69.168.1/crnldata/forgetting/Carla/SleepRecordings"

fc1 = FileChooser(dpath,select_default=True, show_only_dirs = True, title = "<b>Go inside the folder containing the LFP raw file</b>", layout=widgets.Layout(width='100%'))
display(fc1)

# Sample callback function
def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return 

# Register callback function
fc1.register_callback(update_my_folder)

FileChooser(path='\\10.69.168.1\crnldata\forgetting\Carla\SleepRecordings\CarreMarron\7mois', filename='', tit…

Load file, mouse ID & channels

In [ ]:
folder_base = Path(dpath) #Path(dpath).parent
print(folder_base)        
      
#Load signals

LFPfile = Path(f'{folder_base}/DataFrame_rawdataDS.pkl')
LFPs_df = pd.read_pickle(LFPfile)
samplerate = 1000 
numchannel = LFPs_df.shape[1]
rec_ch_list = LFPs_df.columns.values
# Load LFPs timestamps 
for file_pathTS in folder_base.parent.parent.glob('**/continuous/*/timeStampsDS.npy'):
    print('LFPs timestamps file found')
    LFPtimestamps = np.load(file_pathTS)  
print(round(LFPs_df.shape[0]/samplerate/60), 'min of recording')

# Identify mouse & choose threshold for detection
mouse = []
pos_mice = []
for mouse_name in all_LFPcoordinates.index:
    if mouse_name in LFPfile.__str__():
        mouse.append(mouse_name)
        pos_mice.append(LFPfile.__str__().find(mouse_name)) 
mouse = [x for _, x in sorted(zip(pos_mice, mouse))] # sort mouse in the same order as they appear in the path
mouse=mouse[0]   
print(mouse)

# Identify electrodes & create differential LFPs
all_LFPcoordinates = all_LFPcoordinates.astype(str)
for region in all_LFPcoordinates.loc[mouse].index:
    locals()[region] = []
    locals()[f'{region}_ch'] = []
ID=0
RecordedArea=[]
ChoosenChannels=[]
combined0=[]
rec_ch_list_mouse = [value for value in rec_ch_list if 0+(ID*32) <= value <= 31+(ID*32)]
for rec_ch in rec_ch_list_mouse:
    for idx, LFPcoord_str in enumerate(all_LFPcoordinates.loc[mouse]):
        region = all_LFPcoordinates.loc[mouse].index[idx]
        if LFPcoord_str != 'nan':
            LFPcoord = LFPcoord_str.split('_')[:2] # only take into account the 2 first of electrode of that region 
            num_ch = np.where(str(rec_ch-(ID*32)) == np.array(LFPcoord))[0]
            if len(num_ch) > 0:
                region = all_LFPcoordinates.loc[mouse].index[idx]
                LFP = locals()[region]
                LFP = LFP-np.array(LFPs_df[(rec_ch)]) if len(LFP) > 0 else np.array(LFPs_df[(rec_ch)])
                locals()[region] = LFP
                locals()[f'{region}_ch'].append(rec_ch)
                
                break
            continue    
for region in all_LFPcoordinates.loc[mouse].index:
    LFP = locals()[region]
    LFP_ch = locals()[f'{region}_ch']
    if len(LFP) > 0:
        RecordedArea.append(region) 
        ChoosenChannels.append(LFP_ch) 
        combined0 = zscore(LFP[:,np.newaxis]) if len(combined0) == 0 else np.append(combined0, zscore(LFP[:,np.newaxis]), axis=1)
print(RecordedArea)
print(ChoosenChannels)

\\10.69.168.1\crnldata\forgetting\Carla\SleepRecordings\CarreBlanc\7mois
2991 min of recording
CarreBlanc


MemoryError: Unable to allocate 5.35 GiB for an array with shape (179434714, 4) and data type float64

Check signals

In [ ]:
epoch_dur = 5 # define epoch duration in sec
winlen = 30 # default window length in sec

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source =InMemoryAnalogSignalSource(combined0, samplerate, t_start, channel_names=RecordedArea)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['scale_mode'] = 'same_for_all'
colormap = np.insert(['#88FF88', '#8888FF', '#FF8888']* 10, 0, '#FFFFFF')
for idx, ch in enumerate(RecordedArea): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

# FFT
view3 = TimeFreqViewer(source=source, name='FFT')

view3.params['show_axis'] = True
view3.params['timefreq', 'f_start'] = 1
view3.params['timefreq', 'f_stop'] = 30
view3.params['timefreq', 'deltafreq'] = 1 #interval in Hz
view3.params['xsize'] = winlen
for idx, ch in enumerate(RecordedArea):
    if ch == 'EMG': 
        view3.by_channel_params[f'ch{idx}', 'visible'] = False
    else:        
        view3.by_channel_params[f'ch{idx}', 'clim'] = 1
        view3.by_channel_params[f'ch{idx}', 'visible'] = True

#show main window and run Qapp

win.add_view(view1)
win.add_view(view3)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'

0

### Choose EMG channel & Theta channel(s)

In [145]:
ch_nb_EMG = EMG_ch
ch_EMG = EMG

ch_nb_Theta = EnthC_ch
ch_Theta = EnthC

### EMG channel

Filter choosen EMG channel

In [146]:
# Filter parameter :
f_lowcut = 200.
f_hicut = 400.
N = 4
nyq = 0.5 * samplerate
Wn = [f_lowcut/nyq,f_hicut/nyq]  # Nyquist frequency fraction

# Filter creation :
b, a = signal.butter(N, Wn, 'band')
filt_EMG = signal.filtfilt(b, a, ch_EMG)

# EMG amplitude (envelope)
emg_env = np.abs(signal.hilbert(filt_EMG))

# Smooth the envelope over X seconds
window_sec= 1 # in seconds
window_samples = int(window_sec * samplerate)
kernel = np.ones(window_samples) / window_samples
env_EMG = np.convolve(emg_env, kernel, mode='same')


th_EMG=100 if th_EMG is None else th_EMG

Choose best threshold value (move th_EMG)

In [147]:
winlen =  1000 # default window length in sec

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
combined = np.stack([filt_EMG, env_EMG, np.linspace(th_EMG, th_EMG,len(ch_EMG) )], axis=1)
source =InMemoryAnalogSignalSource(combined, samplerate, t_start, channel_names=['EMG_filtered', 'EMG_envelope', 'Threshold'])
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.by_channel_params[f'ch2', 'color'] = "#FF8888"
view1.by_channel_params[f'ch1', 'color'] = "#FFFFFF"
view1.by_channel_params[f'ch0', 'color'] = "#6D6D6D"
view1.auto_scale()


#show main window and run Qapp
win.add_view(view1)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'

0

Decide best th_EMG value

In [159]:
th_EMG=110 # to modify

### Theta channels

Filter choosen Theta channels

In [160]:
# Filter parameter :
f_lowcut = 4
f_hicut = 8
N = 4
nyq = 0.5 * samplerate
Wn = [f_lowcut/nyq,f_hicut/nyq]  # Nyquist frequency fraction

# Filter creation :
b, a = signal.butter(N, Wn, 'band')
filt_Theta = signal.filtfilt(b, a, ch_Theta)

# Theta amplitude (envelope)
theta_env = np.abs(signal.hilbert(filt_Theta))

# Smooth the envelope over X seconds
#window_sec= 30 # in seconds
#window_samples = int(window_sec * samplerate)
#kernel = np.ones(window_samples) / window_samples
#env_Theta = np.convolve(theta_env, kernel, mode='same')

# Smooth the envelope over X seconds
window_sec= 30 # in seconds
window_samples = int(window_sec * samplerate)
from scipy.ndimage import uniform_filter1d
env_Theta = uniform_filter1d(theta_env, size=window_samples)

th_Theta = 650 if th_Theta is None else th_Theta

In [150]:
winlen = 1000 # default window length in sec

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
combined = np.stack([ch_Theta/5, env_Theta, np.linspace(th_Theta, th_Theta, len(ch_Theta) )], axis=1)
source =InMemoryAnalogSignalSource(combined, samplerate, t_start, channel_names=['Theta','Theta_envelope', 'Threshold'])
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.by_channel_params[f'ch2', 'color'] = '#8888FF'
view1.by_channel_params[f'ch1', 'color'] = '#FFFFFF'
view1.by_channel_params[f'ch0', 'color'] = '#6D6D6D'
view1.auto_scale()


# FFT
view3 = TimeFreqViewer(source=source, name='FFT')

view3.params['show_axis'] = True
view3.params['timefreq', 'f_start'] = 1
view3.params['timefreq', 'f_stop'] = 30
view3.params['timefreq', 'deltafreq'] = 1 #interval in Hz
view3.params['xsize'] = winlen
view3.by_channel_params[f'ch0', 'clim'] = 200
view3.by_channel_params[f'ch0', 'visible'] = True

#show main window and run Qapp

win.add_view(view1)
win.add_view(view3)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

viewer has moved already 0 15690.637312 14089.551871999998
viewer has moved already 0 29299.863552 27058.343936
viewer has moved already 0 29299.863552 27378.561024000002
viewer has moved already 0 29299.863552 27538.669567999998
viewer has moved already 0 29299.863552 28018.995199999998
viewer has moved already 0 29299.863552 28179.103743999996
viewer has moved already 0 29299.863552 28499.320831999998
viewer has moved already 0 29299.863552 28819.53792
viewer has moved already 0 29299.863552 28979.646463999998
viewer has moved already 0 29299.863552 29139.755007999996
viewer has moved already 0 30100.406272 29620.08064
viewer has moved already 0 30100.406272 29780.189184
viewer has moved already 0 30100.406272 29940.297727999998
viewer has moved already 0 30580.731904 30260.514816
viewer has moved already 0 30580.731904 30420.623359999998
viewer has moved already 0 31221.16608 30740.840448
viewer has moved already 0 31221.16608 30900.948992
viewer has moved already 0 31221.16608 3106

0

Decide best threshold value

In [155]:
th_Theta=60 # to modify

### Save threshold

In [161]:
names = ['ch_nb_EMG','th_EMG', 'ch_nb_Theta', 'th_Theta']
values = [ch_nb_EMG, th_EMG, ch_nb_Theta, th_Theta]

with open(f"{dpath}/VigState_detection_thresholds.txt", "w") as f:
    for name, val in zip(names, values):
        f.write(f"{name}: {val}\n")

### Process & save scoring

Save scoring

# 4 stages : 'Wake_1', 'NREM_2', 'REM_3', 'undefined_4'

SleepScoring= np.linspace(2, 2, len(env_Theta)) # everything is NREM
SleepScoring[env_Theta>th_Theta] = 3 # if theta is high = REM
SleepScoring[env_EMG>th_EMG] = 1 # if EMG is high = Wake

epoch_duration = 1 # in seconds    
SleepScoring_ds=SleepScoring[::epoch_duration*samplerate]

SleepScoring_df = pd.DataFrame()
SleepScoring_df['time'] = list(range(0, len(SleepScoring_ds) * epoch_duration, epoch_duration))
SleepScoring_df['duration'] = np.linspace(epoch_duration, epoch_duration, len(SleepScoring_ds))
SleepScoring_df['label'] = SleepScoring_ds
SleepScoring_df['label'] = SleepScoring_df['label'].replace({1: 'Wake_1', 2: 'NREM_2', 3: 'REM_3', 4: 'undefined_4'})

file_path = f"{dpath}/EphyViewer_SemiAutoScor{epoch_duration}s.csv"
SleepScoring_df.to_csv(file_path, index=False)
print('Sleep scoring saved')

In [162]:
# 4 stages : 'Wake_1', 'NREM_2', 'REM_3', 'undefined_4'

SleepScoring= np.linspace(2, 2, len(env_Theta)) # everything is NREM
SleepScoring[env_Theta>th_Theta] = 3 # if theta is high = REM
SleepScoring[env_EMG>th_EMG] = 1 # if EMG is high = Wake

epoch_duration = 1 # in seconds    
SleepScoring_ds=SleepScoring[::epoch_duration*samplerate]

# --- Detection of epochs with no signal (EMG=0 and Theta=0 -> undefined_4) + merging of short NREM/REM episodes ---

# AJOUT 1 : détection des epochs sans signal (tout à 0 sur EMG et Theta) -> undefined_4
samples_per_epoch = epoch_duration * samplerate
n_epochs = len(SleepScoring_ds)

# nombre d'epochs COMPLETS réellement disponibles dans env_Theta / env_EMG
n_full_epochs = len(env_Theta) // samples_per_epoch
n_full = n_full_epochs * samples_per_epoch

env_Theta_epochs = env_Theta[:n_full].reshape(n_full_epochs, samples_per_epoch)
env_EMG_epochs   = env_EMG[:n_full].reshape(n_full_epochs, samples_per_epoch)

zero_mask_full = np.all(env_Theta_epochs == 0, axis=1) & np.all(env_EMG_epochs == 0, axis=1)

# on complète jusqu'à n_epochs (le dernier epoch éventuellement incomplet est laissé tel quel, non marqué undefined)
zero_mask = np.zeros(n_epochs, dtype=bool)
zero_mask[:n_full_epochs] = zero_mask_full

SleepScoring_ds[zero_mask] = 4  # aucun signal -> undefined_4

# AJOUT 2 : fusion des épisodes courts (NREM et REM), en excluant les epochs undefined_4
def get_runs(arr):
    runs = []
    start = 0
    for i in range(1, len(arr) + 1):
        if i == len(arr) or arr[i] != arr[start]:
            runs.append([arr[start], start, i])
            start = i
    return runs

def apply_scoring_corrections(arr, epoch_duration):
    arr = arr.copy()
    changed = True
    while changed:
        changed = False
        runs = get_runs(arr)

        # Règle 1 : NREM < 10s encadré par 2 Wake < 30s -> tout en Wake
        for i, (label, start, end) in enumerate(runs):
            duration = (end - start) * epoch_duration
            if label == 2 and duration < 10 and 0 < i < len(runs) - 1:
                prev_label, prev_start, prev_end = runs[i - 1]
                next_label, next_start, next_end = runs[i + 1]
                prev_dur = (prev_end - prev_start) * epoch_duration
                next_dur = (next_end - next_start) * epoch_duration
                if prev_label == 1 and next_label == 1 and prev_dur < 30 and next_dur < 30:
                    arr[start:end] = 1
                    changed = True

        if changed:
            continue

        # Règle 2 : REM < 10s -> dépend des voisins
        runs = get_runs(arr)
        for i, (label, start, end) in enumerate(runs):
            duration = (end - start) * epoch_duration
            if label == 3 and duration < 10 and 0 < i < len(runs) - 1:
                prev_label = runs[i - 1][0]
                next_label = runs[i + 1][0]
                if prev_label == 1 and next_label == 1:
                    arr[start:end] = 1
                    changed = True
                elif prev_label == 2 and next_label == 2:
                    arr[start:end] = 2
                    changed = True
                elif {prev_label, next_label} == {1, 2}:
                    arr[start:end] = 2
                    changed = True
    return arr

# on ne touche pas aux epochs marquées undefined_4 : on corrige, puis on les remet
undefined_positions = np.where(SleepScoring_ds == 4)[0]
SleepScoring_ds = apply_scoring_corrections(SleepScoring_ds, epoch_duration)
SleepScoring_ds[undefined_positions] = 4

SleepScoring_df = pd.DataFrame()
SleepScoring_df['time'] = list(range(0, len(SleepScoring_ds) * epoch_duration, epoch_duration))
SleepScoring_df['duration'] = np.linspace(epoch_duration, epoch_duration, len(SleepScoring_ds))
SleepScoring_df['label'] = SleepScoring_ds
SleepScoring_df['label'] = SleepScoring_df['label'].replace({1: 'Wake_1', 2: 'NREM_2', 3: 'REM_3', 4: 'undefined_4'})

file_path = f"{dpath}/EphyViewer_SemiAutoScor{epoch_duration}s.csv"
SleepScoring_df.to_csv(file_path, index=False)
print('Sleep scoring saved')

Sleep scoring saved


Verify and add undefined stage if necessary

In [163]:
StagesNb = 4 # 4 stages : Wake, NREM, REM, undefined OR 6 stages : AW (Active wake), QW (Quiet wake), NREM, IS (Intermediate sleep), REM, undefined 

labels = ['AW_1', 'QW_2', 'NREM_3', 'IS_4',  'REM_5', 'undefined_6'] if StagesNb == 6 else ['Wake_1', 'NREM_2', 'REM_3', 'undefined_4'] 
epoch_dur = epoch_duration # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = f'{dpath}/EphyViewer_SemiAutoScor{epoch_duration}s.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source =InMemoryAnalogSignalSource(combined0, samplerate, t_start, channel_names=RecordedArea)
view1 = TraceViewer(source=source)

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['scale_mode'] = 'same_for_all'
colormap = np.insert(['#88FF88', '#8888FF', '#FF8888']* 10, 0, '#FFFFFF')
for idx, ch in enumerate(RecordedArea): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

if StagesNb == 6:
    view2.by_label_params['label0', 'color'] = '#ffcd69' #AW
    view2.by_label_params['label1', 'color'] = '#69ffe6' #QW
    view2.by_label_params['label2', 'color'] = '#69cfff' #NREM
    view2.by_label_params['label3', 'color'] = '#cd69ff' #IS
    view2.by_label_params['label4', 'color'] = '#ff69cd' #REM
    view2.by_label_params['label5', 'color'] = '#8f8f8f' #undefined
else: 
    view2.by_label_params['label0', 'color'] = '#ffcd69' #AW
    view2.by_label_params['label1', 'color'] = '#69cfff' #NREM
    view2.by_label_params['label2', 'color'] = '#ff69cd' #REM
    view2.by_label_params['label3', 'color'] = '#8f8f8f' #undefined
view2.params['view_mode'] = 'flat'
view2.controls.hide()

# FFT
view3 = TimeFreqViewer(source=source, name='FFT')

view3.params['show_axis'] = True
view3.params['timefreq', 'f_start'] = 1
view3.params['timefreq', 'f_stop'] = 30
view3.params['timefreq', 'deltafreq'] = 1 #interval in Hz
view3.params['xsize'] = winlen
for idx, ch in enumerate(RecordedArea):
    if ch == 'EMG': 
        view3.by_channel_params[f'ch{idx}', 'visible'] = False
    else:        
        view3.by_channel_params[f'ch{idx}', 'clim'] = 1
        view3.by_channel_params[f'ch{idx}', 'visible'] = True

#show main window and run Qapp

win.add_view(view1)
win.add_view(view3)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
view3_dock = win.viewers['Sleep Scoring']['dock']
view3_dock.setMaximumHeight(120)  # Limit max height
view3_dock.setMinimumHeight(100)   # Set min height
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'

0

# Automatic processing (usefull?)

In [ ]:
# Load LFP coordinates 
Channels = f'{os.getcwd()}/_LFP_coordinates_of_all_mice.csv'
all_LFPcoordinates = pd.read_csv(Channels, index_col=0)

dpath = Path("//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_data/L2_3_mice/")
for txt_file in dpath.rglob("*VigState_detection_thresholds.txt"):
    with open(txt_file, "r") as file:
        
        print(f"Processing file: {txt_file}")

        # Load thresholds
        content = file.read()
        data = {
            f"{k.strip()}": None if v.strip().lower() == "none"
            else ast.literal_eval(v.strip())
            for k, v in (
                line.split(":", 1)
                for line in content.strip().split("\n")
                if ":" in line
            )
        }
        print(data)
        for key, value in data.items():
            locals()[key] = value

        # to be completed